# ViSEF — Extended: Adaptive Clip + Sweep σ + MIA

Chạy **sau** notebook Priority 1 (đã có `runs/p1_fedprox_seed*` và `runs/p1_fedprox_dp_seed*`).

Gồm 3 phần độc lập, có thể chạy riêng:

- **Phần A — Adaptive Clipping:** chạy lại `fedprox_dp` với `--auto_clip`, so sánh với C=1 cố định.
- **Phần B — Sweep σ → ε:** 1 seed × 3 mức ε mục tiêu (1, 2, 3) → đường cong trade-off.
- **Phần C — MIA:** đo AUC rò rỉ trên checkpoint best của 3 cấu hình, chứng minh DP giảm rò rỉ.

Ước lượng: A ~1h, B ~2h, C ~10 phút. Tổng ~3h.

In [ ]:
!pip install -q scikit-learn
import torch, json, subprocess, sys
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
%cd /content
import os
required = ['common.py','client.py','server.py','partition.py','evaluate.py',
            'privacy.py','run_simulation.py','mia_eval.py',
            'prepare_data.sh','dedup_check.py','centralized.py']
missing = [f for f in required if not os.path.exists(f)]
assert not missing, f'Thieu file: {missing}'
print('OK - du file')

In [ ]:
# Cau hinh
QUICK_MODE = False
SEEDS      = [42, 123, 456] if not QUICK_MODE else [42]
ROUNDS     = 20 if not QUICK_MODE else 5
EPOCHS     = 3
BATCH      = 32
LR         = 1e-4
DOM_FRAC   = 0.7
TRAIN_MODE = 'last_blocks'

def sh(cmd):
    print('\n$', ' '.join(map(str,cmd)))
    subprocess.run([str(c) for c in cmd], check=True)

def pick_best_round(run_dir):
    val = json.load(open(Path(run_dir)/'eval'/'eval_val.json'))['results']
    srv = [r for r in val if r['kind']=='server' and r['round']>0]
    return max(srv, key=lambda r: r['macro_f1'])['round']

## Phần A — Adaptive Clipping

Chạy `fedprox_dp` với `--auto_clip` (C tự calibration từ phân vị 90% per-sample grad norm ở 3 batch đầu) và so sánh với C=1 cố định từ Priority 1.

In [ ]:
for seed in SEEDS:
    tag = f'p1_fedprox_dp_autoclip_seed{seed}'
    run_dir = Path('runs')/tag
    if (run_dir/'eval'/'eval_test.json').exists():
        print(f'[SKIP] {tag}'); continue
    print(f'\n========== {tag} ==========')
    cmd = [sys.executable, 'run_simulation.py',
           '--data_root', './brain-tumor-mri-dataset',
           '--tag', tag,
           '--partition_dir', 'partitions',
           '--rounds', str(ROUNDS), '--epochs', str(EPOCHS),
           '--batch_size', str(BATCH), '--lr', str(LR),
           '--train_mode', TRAIN_MODE, '--mu', '0.01',
           '--dominant_frac', str(DOM_FRAC),
           '--seed', str(seed), '--skip_eval',
           '--dp', '--noise_multiplier', '1.0',
           '--auto_clip', '--auto_clip_quantile', '0.9',
           '--verify_dp']
    sh(cmd)
    sh([sys.executable,'evaluate.py','--run_dir',run_dir,
        '--split','val','--partition_dir','partitions',
        '--all_rounds','--device','auto','--no_plots'])
    best = pick_best_round(run_dir)
    print(f'[Pick] {tag}: best round={best}')
    sh([sys.executable,'evaluate.py','--run_dir',run_dir,
        '--split','test','--data_root','./brain-tumor-mri-dataset',
        '--server_rounds',str(best),'--client_rounds','',
        '--device','auto','--no_plots'])

# Aggregate
rows = []
for seed in SEEDS:
    for name, prefix in [('fedprox_dp_C1','p1_fedprox_dp'),
                         ('fedprox_dp_autoclip','p1_fedprox_dp_autoclip')]:
        tag = f'{prefix}_seed{seed}'
        tj = Path('runs')/tag/'eval'/'eval_test.json'
        if not tj.exists(): continue
        t = json.load(open(tj))['results'][0]
        rows.append({'config':name,'seed':seed,
                     'test_acc':t['accuracy'],'test_f1':t['macro_f1']})
df_auto = pd.DataFrame(rows)
print('\n=== Adaptive clipping vs C=1 ===')
print(df_auto.groupby('config')[['test_acc','test_f1']].agg(['mean','std']).round(4).to_string())

## Phần B — Sweep σ → ε mục tiêu 1, 2, 3

Chỉ 1 seed (42) để tiết kiệm thời gian. Mỗi cấu hình dùng `--target_epsilon` để client tự binary search σ. Kết quả là đường cong accuracy–ε — hình quan trọng nhất cho phần privacy.

In [ ]:
SWEEP_SEED   = 42
base_test = Path(f'runs/p1_fedprox_seed{SWEEP_SEED}')/'eval'/'eval_test.json'
assert base_test.exists(), 'Run ViSEF_priority1_multiseed.ipynb first to create the FedProx baseline.'
TARGET_EPS   = [1.0, 2.0, 3.0]
sweep_rows = []
for eps_t in TARGET_EPS:
    tag = f'p2_sweep_eps{eps_t:.1f}_seed{SWEEP_SEED}'
    run_dir = Path('runs')/tag
    if not (run_dir/'eval'/'eval_test.json').exists():
        print(f'\n========== {tag} ==========')
        cmd = [sys.executable,'run_simulation.py',
               '--data_root','./brain-tumor-mri-dataset','--tag',tag,
               '--partition_dir','partitions',
               '--rounds',str(ROUNDS),'--epochs',str(EPOCHS),
               '--batch_size',str(BATCH),'--lr',str(LR),
               '--train_mode',TRAIN_MODE,'--mu','0.01',
               '--dominant_frac',str(DOM_FRAC),
               '--seed',str(SWEEP_SEED),'--skip_eval',
               '--dp','--target_epsilon',str(eps_t),
               '--auto_clip','--verify_dp']
        sh(cmd)
        sh([sys.executable,'evaluate.py','--run_dir',run_dir,
            '--split','val','--partition_dir','partitions',
            '--all_rounds','--device','auto','--no_plots'])
        best = pick_best_round(run_dir)
        sh([sys.executable,'evaluate.py','--run_dir',run_dir,
            '--split','test','--data_root','./brain-tumor-mri-dataset',
            '--server_rounds',str(best),'--client_rounds','',
            '--device','auto','--no_plots'])
    # doc epsilon thuc te tu log aggregate cuoi cung
    logs = sorted((run_dir/'logs').glob('aggregate_round_*.json'))
    eps_real = None
    if logs:
        lg = json.load(open(logs[-1]))
        eps_vals = [v for v in lg.get('dp_epsilon',{}).values() if v and v>0]
        if eps_vals: eps_real = max(eps_vals)
    tj = run_dir/'eval'/'eval_test.json'
    if tj.exists():
        t = json.load(open(tj))['results'][0]
        val = json.load(open(run_dir/'eval'/'eval_val.json'))['results']
        srv = [r for r in val if r['kind']=='server' and r['round']>0]
        best_m = max(srv, key=lambda r:r['macro_f1'])
        sweep_rows.append({'target_eps':eps_t,'real_eps':eps_real,
                           'best_round':best_m['round'],
                           'val_f1':best_m['macro_f1'],
                           'test_acc':t['accuracy'],'test_f1':t['macro_f1']})

df_sweep = pd.DataFrame(sweep_rows)
# them baseline khong DP
base_tag = f'p1_fedprox_seed{SWEEP_SEED}'
bt = Path('runs')/base_tag/'eval'/'eval_test.json'
if bt.exists():
    b = json.load(open(bt))['results'][0]
    df_sweep = pd.concat([pd.DataFrame([{'target_eps':float('inf'),'real_eps':0.0,
        'best_round':None,'val_f1':None,
        'test_acc':b['accuracy'],'test_f1':b['macro_f1']}]), df_sweep],
        ignore_index=True)
df_sweep.to_csv('runs/p2_sweep.csv', index=False)
print('\n=== Sweep σ → ε ===')
print(df_sweep.to_string())

In [ ]:
# Ve duong cong accuracy vs epsilon
fig, ax = plt.subplots(figsize=(7,4.5))
d = df_sweep.dropna(subset=['real_eps']).sort_values('real_eps')
ax.plot(d['real_eps'], d['test_acc']*100, 'o-', label='Test accuracy', lw=2)
ax.plot(d['real_eps'], d['test_f1']*100, 's--', label='Test macro-F1', lw=2)
for _, r in d.iterrows():
    x = r['real_eps']
    ax.annotate(f"ε={x:.2f}",
                (x, r['test_acc']*100), textcoords='offset points',
                xytext=(0,8), ha='center', fontsize=8)
ax.set_xlabel('Epsilon (ε, δ=1e-5)'); ax.set_ylabel('%')
ax.set_title('Trade-off accuracy – privacy (FedProx + DP-SGD, seed 42)')
ax.grid(alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig('runs/p2_sweep_curve.png', dpi=150)
plt.show()

## Phần C — Membership Inference Attack

Đo AUC trên checkpoint best của 3 cấu hình. Kỳ vọng: `fedavg`/`fedprox` (không DP) có AUC cao (rò rỉ), `fedprox_dp` có AUC gần 0,5 hơn (giảm rò rỉ).

In [ ]:
mia_rows = []
for seed in SEEDS:
    for name, prefix in [('fedavg','p1_fedavg'),
                         ('fedprox','p1_fedprox'),
                         ('fedprox_dp','p1_fedprox_dp')]:
        tag = f'{prefix}_seed{seed}'
        run_dir = Path('runs')/tag
        if not (run_dir/'eval'/'eval_val.json').exists(): continue
        best = pick_best_round(run_dir)
        out = run_dir/'mia.json'
        if not out.exists():
            print(f'\n[MIA] {tag} @ round {best}')
            sh([sys.executable,'mia_eval.py',
                '--data_root','./brain-tumor-mri-dataset',
                '--run_dir',str(run_dir),'--round',str(best),
                '--out',str(out),'--device','auto'])
        r = json.load(open(out))
        r['config']=name; r['seed']=seed
        mia_rows.append(r)

df_mia = pd.DataFrame(mia_rows)
df_mia.to_csv('runs/p3_mia.csv', index=False)
print('\n=== MIA AUC (mean ± std tren cac seed) ===')
print(df_mia.groupby('config')[['mia_auc','loss_gap']].agg(['mean','std']).round(4).to_string())

In [ ]:
# Ve histogram AUC
fig, ax = plt.subplots(figsize=(8,4.5))
configs = ['fedavg','fedprox','fedprox_dp']
colors = {'fedavg':'tab:blue','fedprox':'tab:orange','fedprox_dp':'tab:red'}
means = [df_mia[df_mia['config']==c]['mia_auc'].mean() for c in configs]
stds  = [df_mia[df_mia['config']==c]['mia_auc'].std()  for c in configs]
bars = ax.bar(configs, means, yerr=stds, capsize=8,
              color=[colors[c] for c in configs], edgecolor='black')
ax.axhline(0.5, color='gray', ls='--', label='Không rò rỉ (AUC=0.5)')
for b, m, s in zip(bars, means, stds):
    ax.text(b.get_x()+b.get_width()/2, m+ (s if s==s else 0)+0.01,
            f'{m:.3f}±{(s if s==s else 0):.3f}', ha='center', fontsize=9)
ax.set_ylabel('MIA AUC'); ax.set_ylim(0.4, 1.0)
ax.set_title('Membership Inference AUC — DP giảm rò rỉ')
ax.grid(axis='y', alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig('runs/p3_mia.png', dpi=150)
plt.show()

## Xuất file cho báo cáo

Tải các file CSV/PNG để chèn vào LaTeX.

In [ ]:
from google.colab import files
for f in ['runs/p2_sweep.csv','runs/p2_sweep_curve.png',
          'runs/p3_mia.csv','runs/p3_mia.png']:
    if Path(f).exists():
        files.download(f)

# In bang tom tat cho copy vao LaTeX
print('\n=== BANG 1: Sweep σ - ε ===')
print(df_sweep.to_string(index=False))
print('\n=== BANG 2: MIA ===')
print(df_mia.groupby('config')[['mia_auc','mia_attack_acc_best','loss_gap']].agg(['mean','std']).round(4).to_string())
if 'df_auto' in globals():
    print('\n=== BANG 3: Adaptive clip vs C=1 ===')
    print(df_auto.groupby('config')[['test_acc','test_f1']].agg(['mean','std']).round(4).to_string())